<a href="https://colab.research.google.com/github/PSD-20/Portfolio-Optimization/blob/main/Portfolio_VQE_4_Qubit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
# ============================================================
# CELL 1 — Imports and Configuration
# Quantum Portfolio Optimization using VQE
# ============================================================

# -----------------------------
# Standard library
# -----------------------------
from dataclasses import dataclass
from itertools import product
from typing import Optional, Callable, Dict, List, Tuple, Any


# -----------------------------
# Numerical / data libraries
# -----------------------------
import numpy as np
import pandas as pd

from scipy.optimize import minimize


# -----------------------------
# Visualization
# -----------------------------
import matplotlib.pyplot as plt


# -----------------------------
# Qiskit
# -----------------------------
!pip install qiskit

!pip install qiskit-aer
import qiskit

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector

from qiskit.quantum_info import (
    SparsePauliOp,
    Statevector
)

from qiskit.primitives import (
    StatevectorEstimator,
    StatevectorSampler
)


# ============================================================
# Reproducibility
# ============================================================

RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)


# ============================================================
# Global numerical settings
# ============================================================

NUMERICAL_TOLERANCE = 1e-8

np.set_printoptions(
    precision=6,
    suppress=True
)

pd.set_option(
    "display.float_format",
    lambda x: f"{x:.8f}"
)


# ============================================================
# Toy Portfolio Configuration
# ============================================================

NUM_ASSETS = 5

# Asset names
ASSET_NAMES = [
    "A", "B", "C", "D",
]

MU = np.array([
    0.08,
    0.12,
    0.15,
    0.07,
])

# Covariance matrix
SIGMA = np.array([
  [0.04,  0.01,  0.005, 0.002],
  [0.01,  0.07,  0.015, 0.003],
  [0.01,  0.07,  0.015, 0.003],
  [0.002, 0.003, 0.004, 0.03]
])

# Markowitz parameters
GAMMA = 1.0
BUDGET_PENALTY = 100.0


INVESTMENT = np.array([
    1.0,   # A
    2.0,   # B
    3.0,   # C
    1.0,   # D
])

# Total investment budget
BUDGET = 5.0


# ============================================================
# VQE Configuration
# ============================================================

ANSATZ_LAYERS = 4

ROTATION_GATES = ("ry", "rz")

ENTANGLEMENT = "full"

OPTIMIZER = "COBYLA"

MAX_ITERATIONS = 1000

VQE_TOLERANCE = 1e-8

INITIAL_PARAMETERS = "random"

SHOTS = 4096


# ============================================================
# Display configuration
# ============================================================

print("Environment configured successfully.")
print(f"Qiskit version        : {qiskit.__version__}")
print(f"Random seed            : {RANDOM_SEED}")
print(f"Number of assets       : {NUM_ASSETS}")
print(f"Ansatz layers          : {ANSATZ_LAYERS}")
print(f"Optimizer              : {OPTIMIZER}")
print(f"Maximum iterations     : {MAX_ITERATIONS}")
print(f"VQE tolerance           : {VQE_TOLERANCE}")
print(f"Sampling shots         : {SHOTS}")

Environment configured successfully.
Qiskit version        : 2.5.2
Random seed            : 42
Number of assets       : 5
Ansatz layers          : 4
Optimizer              : COBYLA
Maximum iterations     : 1000
VQE tolerance           : 1e-08
Sampling shots         : 4096


In [15]:
# ============================================================
# CELL 2 — Portfolio Problem Definition
# ============================================================

@dataclass
class PortfolioProblem:
    """
    Binary portfolio optimization problem.

    Each asset has a binary decision variable:

        x_i = 1 -> asset selected
        x_i = 0 -> asset not selected

    Investment constraint:

        investment^T x = budget

    Markowitz objective:

        -mu^T x + gamma * x^T Sigma x

    Full penalized objective:

        C(x) =
            -mu^T x
            + gamma * x^T Sigma x
            + penalty * (investment^T x - budget)^2
    """

    mu: np.ndarray
    Sigma: np.ndarray
    investment: np.ndarray
    budget: float
    gamma: float
    penalty: float
    asset_names: Optional[List[str]] = None

    def __post_init__(self):

        self.mu = np.asarray(
            self.mu,
            dtype=float
        )

        self.Sigma = np.asarray(
            self.Sigma,
            dtype=float
        )

        self.investment = np.asarray(
            self.investment,
            dtype=float
        )

        # ----------------------------------------------------
        # Expected returns
        # ----------------------------------------------------

        if self.mu.ndim != 1:
            raise ValueError(
                "`mu` must be one-dimensional."
            )

        self.num_assets = len(self.mu)

        # ----------------------------------------------------
        # Covariance matrix
        # ----------------------------------------------------

        if self.Sigma.shape != (
            self.num_assets,
            self.num_assets
        ):
            raise ValueError(
                "`Sigma` must have shape "
                f"({self.num_assets}, {self.num_assets})."
            )

        # ----------------------------------------------------
        # Investment vector
        # ----------------------------------------------------

        if self.investment.shape != (
            self.num_assets,
        ):
            raise ValueError(
                "`investment` must contain one value "
                "for every asset."
            )

        # ----------------------------------------------------
        # Parameters
        # ----------------------------------------------------

        if self.gamma < 0:
            raise ValueError(
                "`gamma` must be non-negative."
            )

        if self.penalty < 0:
            raise ValueError(
                "`penalty` must be non-negative."
            )

        if self.budget < 0:
            raise ValueError(
                "`budget` must be non-negative."
            )

        # ----------------------------------------------------
        # Asset names
        # ----------------------------------------------------

        if self.asset_names is None:
            self.asset_names = [
                f"Asset_{i}"
                for i in range(self.num_assets)
            ]
        else:
            if len(self.asset_names) != self.num_assets:
                raise ValueError(
                    "`asset_names` must have one value "
                    "for every asset."
                )

            self.asset_names = list(
                self.asset_names
            )

    def summary(self):

        print("=" * 60)
        print("Portfolio Problem")
        print("=" * 60)

        print(
            f"Number of assets : {self.num_assets}"
        )

        print(
            f"Budget           : {self.budget}"
        )

        print(
            f"Risk aversion    : {self.gamma}"
        )

        print(
            f"Budget penalty   : {self.penalty}"
        )

        print("\nExpected Returns:")

        print(
            pd.Series(
                self.mu,
                index=self.asset_names,
                name="mu"
            )
        )

        print("\nInvestment Values:")

        print(
            pd.Series(
                self.investment,
                index=self.asset_names,
                name="investment"
            )
        )

        print("\nCovariance Matrix:")

        print(
            pd.DataFrame(
                self.Sigma,
                index=self.asset_names,
                columns=self.asset_names
            )
        )

        print("=" * 60)

In [16]:
# ============================================================
# 4-Asset Toy Portfolio
# ============================================================

portfolio = PortfolioProblem(
    mu=MU,
    Sigma=SIGMA,
    investment=INVESTMENT,
    budget=BUDGET,
    gamma=GAMMA,
    penalty=BUDGET_PENALTY,
    asset_names=ASSET_NAMES
)

In [17]:
# ============================================================
# CELL 3 — Classical Portfolio Cost Function
# ============================================================

def calculate_portfolio_cost(
    x: np.ndarray,
    problem: PortfolioProblem
) -> Dict[str, Any]:
    """
    Calculate the portfolio objective.

    Markowitz objective:

        -mu^T x + gamma * x^T Sigma x

    Budget constraint:

        investment^T x = budget

    Penalized objective:

        C(x) =
            -mu^T x
            + gamma * x^T Sigma x
            + penalty * (investment^T x - budget)^2
    """

    x = np.asarray(
        x,
        dtype=int
    )

    # --------------------------------------------------------
    # Validate x
    # --------------------------------------------------------

    if x.shape != (
        problem.num_assets,
    ):
        raise ValueError(
            f"`x` must have shape "
            f"({problem.num_assets},)."
        )

    if not np.all(
        np.isin(x, [0, 1])
    ):
        raise ValueError(
            "`x` must contain only 0 and 1."
        )

    # --------------------------------------------------------
    # Expected return
    # --------------------------------------------------------

    expected_return = float(
        problem.mu @ x
    )

    # --------------------------------------------------------
    # Risk
    # --------------------------------------------------------

    risk = float(
        x @ problem.Sigma @ x
    )

    # --------------------------------------------------------
    # Investment used
    #
    # investment^T x
    # --------------------------------------------------------

    investment_used = float(
        problem.investment @ x
    )

    # --------------------------------------------------------
    # Budget constraint
    #
    # investment^T x - budget
    # --------------------------------------------------------

    budget_difference = (
        investment_used
        - problem.budget
    )

    # --------------------------------------------------------
    # Budget penalty
    # --------------------------------------------------------

    penalty_cost = float(
        problem.penalty
        * budget_difference ** 2
    )

    # --------------------------------------------------------
    # Markowitz objective
    # --------------------------------------------------------

    markowitz_objective = float(
        -expected_return
        + problem.gamma * risk
    )

    # --------------------------------------------------------
    # Complete QUBO objective
    # --------------------------------------------------------

    total_cost = float(
        markowitz_objective
        + penalty_cost
    )

    selected_assets = [
        problem.asset_names[i]
        for i, bit in enumerate(x)
        if bit == 1
    ]

    return {
        "expected_return": expected_return,
        "risk": risk,
        "investment": investment_used,
        "budget_difference": budget_difference,
        "penalty": penalty_cost,
        "markowitz_objective": markowitz_objective,
        "total_cost": total_cost,
        "selected_count": int(np.sum(x)),
        "selected_assets": ", ".join(
            selected_assets
        ) if selected_assets else "None"
    }

In [18]:
# ============================================================
# Test Portfolio
# ============================================================

x_test = np.array([
    1, 0, 1, 1,

])

cost_test = calculate_portfolio_cost(
    x=x_test,
    problem=portfolio
)

print("Portfolio:", x_test)
print()

print(
    f"Selected assets     : "
    f"{cost_test['selected_assets']}"
)

print(
    f"Expected return     : "
    f"{cost_test['expected_return']:.6f}"
)

print(
    f"Risk                : "
    f"{cost_test['risk']:.6f}"
)

print(
    f"Investment          : "
    f"{cost_test['investment']:.6f}"
)

print(
    f"Budget difference    : "
    f"{cost_test['budget_difference']:.6f}"
)

print(
    f"Penalty             : "
    f"{cost_test['penalty']:.6f}"
)

print(
    f"Markowitz objective  : "
    f"{cost_test['markowitz_objective']:.6f}"
)

print(
    f"Total objective      : "
    f"{cost_test['total_cost']:.6f}"
)

Portfolio: [1 0 1 1]

Selected assets     : A, C, D
Expected return     : 0.300000
Risk                : 0.111000
Investment          : 5.000000
Budget difference    : 0.000000
Penalty             : 0.000000
Markowitz objective  : -0.189000
Total objective      : -0.189000


In [19]:
# ============================================================
# CELL 4 — Exact Classical Solver
# ============================================================

# ============================================================
# Exact Classical Solver
# ============================================================

def solve_classically(
    problem: PortfolioProblem,
    sort_results: bool = True
) -> pd.DataFrame:
    """
    Exhaustively evaluate all 2^N binary portfolios.
    """

    results = []

    for bits in product(
        [0, 1],
        repeat=problem.num_assets
    ):

        x = np.array(
            bits,
            dtype=int
        )

        cost_data = calculate_portfolio_cost(
            x=x,
            problem=problem
        )

        results.append({
            "bitstring":
                "".join(map(str, x)),

            "selected_assets":
                cost_data["selected_assets"],

            "expected_return":
                cost_data["expected_return"],

            "risk":
                cost_data["risk"],

            "investment":
                cost_data["investment"],

            "budget_difference":
                cost_data["budget_difference"],

            "penalty":
                cost_data["penalty"],

            "markowitz_objective":
                cost_data["markowitz_objective"],

            "objective":
                cost_data["total_cost"],

            "selected_count":
                cost_data["selected_count"]
        })

    results_df = pd.DataFrame(
        results
    )

    if sort_results:

        results_df = (
            results_df
            .sort_values(
                by="objective",
                ascending=True
            )
            .reset_index(drop=True)
        )

    return results_df


def get_classical_optimum(
    classical_results: pd.DataFrame
) -> Dict[str, Any]:
    """
    Return the lowest-cost portfolio from
    the exhaustive search.
    """

    optimum = classical_results.iloc[0]

    return {
        "bitstring":
            optimum["bitstring"],

        "selected_assets":
            optimum["selected_assets"],

        "expected_return":
            float(
                optimum["expected_return"]
            ),

        "risk":
            float(
                optimum["risk"]
            ),

        "investment":
            float(
                optimum["investment"]
            ),

        "budget_difference":
            float(
                optimum["budget_difference"]
            ),

        "penalty":
            float(
                optimum["penalty"]
            ),

        "markowitz_objective":
            float(
                optimum["markowitz_objective"]
            ),

        "selected_count":
            int(
                optimum["selected_count"]
            ),

        "objective":
            float(
                optimum["objective"]
            )
    }

In [20]:
# ============================================================
# Run Exact Classical Search
# ============================================================

classical_results = solve_classically(
    problem=portfolio
)

print(
    "Number of portfolios evaluated:",
    len(classical_results)
)

display(
    classical_results
)

# ============================================================
# Exact Classical Optimum
# ============================================================

classical_optimum = get_classical_optimum(
    classical_results
)

print("=" * 60)
print("EXACT CLASSICAL OPTIMUM")
print("=" * 60)

print(
    f"Bitstring          : "
    f"{classical_optimum['bitstring']}"
)

print(
    f"Selected assets    : "
    f"{classical_optimum['selected_assets']}"
)

print(
    f"Expected return    : "
    f"{classical_optimum['expected_return']:.6f}"
)

print(
    f"Risk               : "
    f"{classical_optimum['risk']:.6f}"
)

print(
    f"Investment         : "
    f"{classical_optimum['investment']:.6f}"
)

print(
    f"Budget difference  : "
    f"{classical_optimum['budget_difference']:.6f}"
)

print(
    f"Penalty            : "
    f"{classical_optimum['penalty']:.6f}"
)

print(
    f"Selected count     : "
    f"{classical_optimum['selected_count']}"
)

print(
    f"Objective          : "
    f"{classical_optimum['objective']:.6f}"
)

Number of portfolios evaluated: 16


,bitstring,selected_assets,expected_return,risk,investment,budget_difference,penalty,markowitz_objective,objective,selected_count
0,1011,"A, C, D",0.30000000,0.11100000,5.00000000,0.00000000,0.00000000,-0.18900000,-0.18900000,3
1,0110,"B, C",0.27000000,0.17000000,5.00000000,0.00000000,0.00000000,-0.10000000,-0.10000000,2
2,0011,"C, D",0.22000000,0.05200000,4.00000000,-1.00000000,100.00000000,-0.16800000,99.83200000,2
3,1010,"A, C",0.23000000,0.07000000,4.00000000,-1.00000000,100.00000000,-0.16000000,99.84000000,2
4,0111,"B, C, D",0.34000000,0.21300000,6.00000000,1.00000000,100.00000000,-0.12700000,99.87300000,3
5,1110,"A, B, C",0.35000000,0.24500000,6.00000000,1.00000000,100.00000000,-0.10500000,99.89500000,3
6,1101,"A, B, D",0.27000000,0.17000000,4.00000000,-1.00000000,100.00000000,-0.10000000,99.90000000,3
7,0010,C,0.15000000,0.01500000,3.00000000,-2.00000000,400.00000000,-0.13500000,399.86500000,1
8,1111,"A, B, C, D",0.42000000,0.29200000,7.00000000,2.00000000,400.00000000,-0.12800000,399.87200000,4
9,0101,"B, D",0.19000000,0.10600000,3.00000000,-2.00000000,400.00000000,-0.08400000,399.91600000,2


EXACT CLASSICAL OPTIMUM
Bitstring          : 1011
Selected assets    : A, C, D
Expected return    : 0.300000
Risk               : 0.111000
Investment         : 5.000000
Budget difference  : 0.000000
Penalty            : 0.000000
Selected count     : 3
Objective          : -0.189000


In [21]:
# ============================================================
# Feasible Portfolios Only
# ============================================================

feasible_results = classical_results[
    np.isclose(
        classical_results["investment"],
        portfolio.budget
    )
].copy()

print(
    f"Number of feasible portfolios: "
    f"{len(feasible_results)}"
)

display(
    feasible_results
)

# ============================================================
# Classical Solver Sanity Checks
# ============================================================

expected_number_of_portfolios = (
    2 ** portfolio.num_assets
)

assert len(classical_results) == (
    expected_number_of_portfolios
)

print(
    "Classical exhaustive-search validation passed."
)

print(
    f"Total portfolios evaluated : "
    f"{len(classical_results)}"
)

print(
    f"Feasible portfolios        : "
    f"{len(feasible_results)}"
)

Number of feasible portfolios: 2


,bitstring,selected_assets,expected_return,risk,investment,budget_difference,penalty,markowitz_objective,objective,selected_count
0,1011,"A, C, D",0.30000000,0.11100000,5.00000000,0.00000000,0.00000000,-0.18900000,-0.18900000,3
1,0110,"B, C",0.27000000,0.17000000,5.00000000,0.00000000,0.00000000,-0.10000000,-0.10000000,2


Classical exhaustive-search validation passed.
Total portfolios evaluated : 16
Feasible portfolios        : 2


In [22]:
# ============================================================
# CELL 5 — Direct Cost Hamiltonian Construction
# ============================================================

# ============================================================
# Budget-Constrained Cost Hamiltonian
# ============================================================

def _z_pauli_label(
    qubit_index: int,
    num_qubits: int
) -> str:

    label = ["I"] * num_qubits

    label[
        num_qubits - 1 - qubit_index
    ] = "Z"

    return "".join(label)


def _zz_pauli_label(
    qubit_i: int,
    qubit_j: int,
    num_qubits: int
) -> str:

    label = ["I"] * num_qubits

    label[
        num_qubits - 1 - qubit_i
    ] = "Z"

    label[
        num_qubits - 1 - qubit_j
    ] = "Z"

    return "".join(label)


# ============================================================
# Build Cost Hamiltonian
# ============================================================

def build_cost_hamiltonian(
    problem: PortfolioProblem
) -> Tuple[
    SparsePauliOp,
    float,
    np.ndarray,
    np.ndarray
]:
    """
    Build the Ising Hamiltonian for

        C(x) =
            -mu^T x
            + gamma * x^T Sigma x
            + penalty * (investment^T x - budget)^2

    using

        x_i = (1 - Z_i) / 2

    IMPORTANT:
    Sigma supplied by the user is not symmetric.

    Since

        x^T Sigma x

    depends only on the symmetric part of Sigma,

        Sigma_eff = (Sigma + Sigma.T) / 2

    is used implicitly.

    Therefore, for i < j, the risk cross-term is

        gamma * (Sigma[i,j] + Sigma[j,i]).
    """

    n = problem.num_assets

    gamma = problem.gamma
    penalty = problem.penalty

    investment = problem.investment
    budget = problem.budget

    # ========================================================
    # 1. BINARY QUBO COEFFICIENTS
    #
    # C(x) =
    #     constant
    #     + sum_i a_i x_i
    #     + sum_{i<j} b_ij x_i x_j
    # ========================================================

    # --------------------------------------------------------
    # Linear coefficients
    #
    # From Markowitz:
    #
    #     gamma * Sigma_ii - mu_i
    #
    # From budget penalty:
    #
    #     penalty * (w_i^2 - 2 B w_i)
    # --------------------------------------------------------

    linear_coefficients = (
        gamma * np.diag(problem.Sigma)
        - problem.mu
        + penalty
        * (
            investment ** 2
            - 2.0 * budget * investment
        )
    )

    # --------------------------------------------------------
    # Pair coefficients
    #
    # Risk contribution:
    #
    #     gamma * (Sigma_ij + Sigma_ji)
    #
    # Budget contribution:
    #
    #     2 * penalty * w_i * w_j
    # --------------------------------------------------------

    pair_coefficients = np.zeros(
        (n, n),
        dtype=float
    )

    for i in range(n):

        for j in range(i + 1, n):

            risk_pair = (
                gamma
                * (
                    problem.Sigma[i, j]
                    + problem.Sigma[j, i]
                )
            )

            budget_pair = (
                2.0
                * penalty
                * investment[i]
                * investment[j]
            )

            pair_coefficients[i, j] = (
                risk_pair
                + budget_pair
            )

            pair_coefficients[j, i] = (
                risk_pair
                + budget_pair
            )

    # ========================================================
    # 2. BINARY -> ISING
    #
    # x_i = (1 - Z_i)/2
    #
    # x_i x_j =
    #     (1 - Z_i - Z_j + Z_i Z_j)/4
    # ========================================================

    h_coefficients = np.zeros(
        n,
        dtype=float
    )

    interaction_coefficients = np.zeros(
        (n, n),
        dtype=float
    )

    # --------------------------------------------------------
    # Constant term from budget penalty
    #
    #     penalty * B^2
    # --------------------------------------------------------

    constant = (
        penalty * budget ** 2
    )

    # --------------------------------------------------------
    # Transform linear binary terms
    # --------------------------------------------------------

    for i in range(n):

        a_i = linear_coefficients[i]

        constant += (
            a_i / 2.0
        )

        h_coefficients[i] += (
            -a_i / 2.0
        )

    # --------------------------------------------------------
    # Transform quadratic binary terms
    # --------------------------------------------------------

    for i in range(n):

        for j in range(i + 1, n):

            b_ij = pair_coefficients[i, j]

            constant += (
                b_ij / 4.0
            )

            h_coefficients[i] += (
                -b_ij / 4.0
            )

            h_coefficients[j] += (
                -b_ij / 4.0
            )

            interaction_coefficients[i, j] = (
                b_ij / 4.0
            )

            interaction_coefficients[j, i] = (
                b_ij / 4.0
            )

    # ========================================================
    # 3. CREATE QISKIT HAMILTONIAN
    # ========================================================

    pauli_terms = []

    # --------------------------------------------------------
    # Identity
    # --------------------------------------------------------

    pauli_terms.append(
        (
            "I" * n,
            float(constant)
        )
    )

    # --------------------------------------------------------
    # Z terms
    # --------------------------------------------------------

    for i in range(n):

        coefficient = (
            h_coefficients[i]
        )

        if not np.isclose(
            coefficient,
            0.0,
            atol=NUMERICAL_TOLERANCE
        ):

            pauli_terms.append(
                (
                    _z_pauli_label(i, n),
                    coefficient
                )
            )

    # --------------------------------------------------------
    # ZZ terms
    # --------------------------------------------------------

    for i in range(n):

        for j in range(i + 1, n):

            coefficient = (
                interaction_coefficients[i, j]
            )

            if not np.isclose(
                coefficient,
                0.0,
                atol=NUMERICAL_TOLERANCE
            ):

                pauli_terms.append(
                    (
                        _zz_pauli_label(i, j, n),
                        coefficient
                    )
                )

    # --------------------------------------------------------
    # SparsePauliOp
    # --------------------------------------------------------

    hamiltonian = (
        SparsePauliOp.from_list(
            pauli_terms
        ).simplify()
    )

    return (
        hamiltonian,
        float(constant),
        h_coefficients,
        interaction_coefficients
    )

In [23]:
# ============================================================
# Build Cost Hamiltonian
# ============================================================

(
    cost_hamiltonian,
    hamiltonian_constant,
    h_coefficients,
    J_coefficients
) = build_cost_hamiltonian(
    problem=portfolio
)

print("Cost Hamiltonian:")
print(cost_hamiltonian)

print("\nIdentity coefficient:")
print(
    f"{hamiltonian_constant:.8f}"
)

print("\nSingle-qubit Z coefficients:")

for i, coefficient in enumerate(
    h_coefficients
):

    print(
        f"q{i} "
        f"({portfolio.asset_names[i]}): "
        f"{coefficient:.8f}"
    )

print("\nZZ interaction coefficients:")

display(
    pd.DataFrame(
        J_coefficients,
        index=portfolio.asset_names,
        columns=portfolio.asset_names
    )
)

Cost Hamiltonian:
SparsePauliOp(['IIII', 'IIIZ', 'IIZI', 'IZII', 'ZIII', 'IIZZ', 'IZIZ', 'ZIIZ', 'IZZI', 'ZIZI', 'ZZII'],
              coeffs=[599.90175+0.j, 150.01025+0.j, 299.99725+0.j, 450.04075+0.j, 150.01575+0.j,
 100.005  +0.j, 150.00375+0.j,  50.001  +0.j, 300.02125+0.j, 100.0015 +0.j,
 150.00175+0.j])

Identity coefficient:
599.90175000

Single-qubit Z coefficients:
q0 (A): 150.01025000
q1 (B): 299.99725000
q2 (C): 450.04075000
q3 (D): 150.01575000

ZZ interaction coefficients:


,A,B,C,D
A,0.00000000,100.00500000,150.00375000,50.00100000
B,100.00500000,0.00000000,300.02125000,100.00150000
C,150.00375000,300.02125000,0.00000000,150.00175000
D,50.00100000,100.00150000,150.00175000,0.00000000


In [24]:
# ============================================================
# CELL 6 — Hamiltonian Validation
# ============================================================

def validate_hamiltonian(
    problem: PortfolioProblem,
    hamiltonian: SparsePauliOp,
    tolerance: float = NUMERICAL_TOLERANCE
) -> pd.DataFrame:
    """
    Validate that the cost Hamiltonian reproduces the
    classical portfolio objective for every binary state.

    Required relationship:

        <x|H_C|x> = C(x)

    Parameters
    ----------
    problem : PortfolioProblem
        Portfolio problem definition.

    hamiltonian : SparsePauliOp
        Ising cost Hamiltonian.

    tolerance : float
        Numerical tolerance for validation.

    Returns
    -------
    pd.DataFrame
        Validation table for all computational-basis states.

    Raises
    ------
    ValueError
        If any Hamiltonian energy differs from the corresponding
        classical cost by more than the specified tolerance.
    """

    n = problem.num_assets

    # Convert the SparsePauliOp into a dense matrix.
    #
    # This is perfectly reasonable for the current 4-qubit
    # toy problem and makes the validation transparent.
    H_matrix = hamiltonian.to_matrix()

    validation_rows = []

    # --------------------------------------------------------
    # Check every computational-basis state
    # --------------------------------------------------------

    for bits in product([0, 1], repeat=n):

        x = np.array(bits, dtype=int)

        # Classical objective
        classical_data = calculate_portfolio_cost(
            x=x,
            problem=problem
        )

        classical_cost = classical_data["total_cost"]

        # ----------------------------------------------------
        # Qiskit's computational basis convention
        #
        # Statevector basis index uses:
        #
        # |q_(n-1) ... q_1 q_0>
        #
        # while our x vector is stored as:
        #
        # [x_0, x_1, ..., x_(n-1)]
        #
        # Therefore reverse the bitstring when converting
        # to a computational-basis matrix index.
        # ----------------------------------------------------

        bitstring = "".join(map(str, x))

        qiskit_bitstring = bitstring[::-1]

        basis_index = int(
            qiskit_bitstring,
            2
        )

        basis_state = np.zeros(
            2 ** n,
            dtype=complex
        )

        basis_state[basis_index] = 1.0

        # Hamiltonian energy:
        #
        # <x|H|x>
        hamiltonian_energy = np.vdot(
            basis_state,
            H_matrix @ basis_state
        ).real

        difference = (
            hamiltonian_energy
            - classical_cost
        )

        validation_rows.append({
            "bitstring": bitstring,
            "classical_cost": classical_cost,
            "hamiltonian_energy": hamiltonian_energy,
            "difference": difference,
            "valid": np.isclose(
                hamiltonian_energy,
                classical_cost,
                atol=tolerance
            )
        })

    validation_df = pd.DataFrame(
        validation_rows
    )

    # --------------------------------------------------------
    # Overall validation
    # --------------------------------------------------------

    max_error = np.max(
        np.abs(validation_df["difference"])
    )

    all_valid = bool(
        np.all(validation_df["valid"])
    )

    print("=" * 65)
    print("HAMILTONIAN VALIDATION")
    print("=" * 65)

    print(
        f"States tested : {len(validation_df)}"
    )

    print(
        f"Maximum error : {max_error:.12e}"
    )

    print(
        f"Tolerance     : {tolerance:.12e}"
    )

    print(
        f"Validation    : {'PASSED' if all_valid else 'FAILED'}"
    )

    print("=" * 65)

    if not all_valid:
        raise ValueError(
            "Hamiltonian validation failed. "
            "The Hamiltonian does not reproduce the "
            "classical portfolio objective within tolerance."
        )

    return validation_df

In [25]:
# ============================================================
# Run Hamiltonian Validation
# ============================================================

hamiltonian_validation = validate_hamiltonian(
    problem=portfolio,
    hamiltonian=cost_hamiltonian
)

display(
    hamiltonian_validation
)

HAMILTONIAN VALIDATION
States tested : 16
Maximum error : 4.547473508865e-13
Tolerance     : 1.000000000000e-08
Validation    : PASSED


,bitstring,classical_cost,hamiltonian_energy,difference,valid
0,0000,2500.00000000,2500.00000000,0.00000000,True
1,0001,1599.96000000,1599.96000000,0.00000000,True
2,0010,399.86500000,399.86500000,-0.00000000,True
3,0011,99.83200000,99.83200000,-0.00000000,True
4,0100,899.95000000,899.95000000,0.00000000,True
5,0101,399.91600000,399.91600000,-0.00000000,True
6,0110,-0.10000000,-0.10000000,-0.00000000,True
7,0111,99.87300000,99.87300000,-0.00000000,True
8,1000,1599.96000000,1599.96000000,0.00000000,True
9,1001,899.92400000,899.92400000,-0.00000000,True


In [26]:
# ============================================================
# Validation Summary
# ============================================================

assert len(hamiltonian_validation) == 2 ** portfolio.num_assets

assert np.all(
    hamiltonian_validation["valid"]
)

print(
    "\nAll computational-basis states reproduce the "
    "classical portfolio objective."
)


All computational-basis states reproduce the classical portfolio objective.


In [27]:
# ============================================================
# CELL 7 — Configurable VQE Ansatz
# ============================================================

def build_vqe_ansatz(
    num_qubits: int,
    num_layers: int = 2,
    rotation_gates: Tuple[str, ...] = ("ry",),
    entanglement: str = "linear"
) -> Tuple[QuantumCircuit, ParameterVector]:
    """
    Build a configurable hardware-efficient VQE ansatz.

    Structure for each layer:

        Single-qubit rotations
                 ↓
        Entangling gates

    Parameters
    ----------
    num_qubits : int
        Number of qubits.

    num_layers : int
        Number of variational layers.

    rotation_gates : tuple of str
        Rotation gates to apply to each qubit.
        Supported:
            "rx"
            "ry"
            "rz"

        For example:
            ("ry",)
            ("ry", "rz")

    entanglement : str
        Entanglement pattern.

        Supported:
            "linear"
            "ring"
            "full"
            "none"

    Returns
    -------
    circuit : QuantumCircuit
        Parameterized ansatz circuit.

    parameters : ParameterVector
        Variational parameters used by the circuit.
    """

    if num_qubits < 1:
        raise ValueError(
            "`num_qubits` must be at least 1."
        )

    if num_layers < 1:
        raise ValueError(
            "`num_layers` must be at least 1."
        )

    supported_rotations = {"rx", "ry", "rz"}

    invalid_rotations = (
        set(rotation_gates) - supported_rotations
    )

    if invalid_rotations:
        raise ValueError(
            f"Unsupported rotation gates: {invalid_rotations}. "
            f"Supported gates: {supported_rotations}"
        )

    supported_entanglement = {
        "linear",
        "ring",
        "full",
        "none"
    }

    if entanglement not in supported_entanglement:
        raise ValueError(
            f"Unsupported entanglement pattern: "
            f"{entanglement}. "
            f"Supported patterns: {supported_entanglement}"
        )

    # --------------------------------------------------------
    # Number of parameters
    # --------------------------------------------------------

    parameters_per_layer = (
        num_qubits * len(rotation_gates)
    )

    total_parameters = (
        num_layers * parameters_per_layer
    )

    parameters = ParameterVector(
        "theta",
        length=total_parameters
    )

    circuit = QuantumCircuit(
        num_qubits,
        name="VQE_Ansatz"
    )

    parameter_index = 0

    # --------------------------------------------------------
    # Build variational layers
    # --------------------------------------------------------

    for layer in range(num_layers):

        # --------------------------------------------
        # Single-qubit rotation layer
        # --------------------------------------------

        for gate_name in rotation_gates:

            for qubit in range(num_qubits):

                theta = parameters[
                    parameter_index
                ]

                if gate_name == "rx":
                    circuit.rx(theta, qubit)

                elif gate_name == "ry":
                    circuit.ry(theta, qubit)

                elif gate_name == "rz":
                    circuit.rz(theta, qubit)

                parameter_index += 1

        # --------------------------------------------
        # Entangling layer
        # --------------------------------------------

        if entanglement == "linear":

            for qubit in range(num_qubits - 1):
                circuit.cx(
                    qubit,
                    qubit + 1
                )

        elif entanglement == "ring":

            for qubit in range(num_qubits - 1):
                circuit.cx(
                    qubit,
                    qubit + 1
                )

            if num_qubits > 2:
                circuit.cx(
                    num_qubits - 1,
                    0
                )

        elif entanglement == "full":

            for control in range(num_qubits):
                for target in range(control + 1, num_qubits):
                    circuit.cx(
                        control,
                        target
                    )

        # No gates for "none"

    return circuit, parameters

In [28]:
# ============================================================
# Build the Toy-Problem VQE Ansatz
# ============================================================

ansatz, ansatz_parameters = build_vqe_ansatz(
    num_qubits=portfolio.num_assets,
    num_layers=ANSATZ_LAYERS,
    rotation_gates=ROTATION_GATES,
    entanglement=ENTANGLEMENT
)

print("Number of qubits:", ansatz.num_qubits)
print("Number of parameters:", len(ansatz_parameters))
print()
print(ansatz)

Number of qubits: 4
Number of parameters: 32

     ┌──────────────┐┌──────────────┐               ┌──────────────┐»
q_0: ┤ Ry(theta[0]) ├┤ Rz(theta[4]) ├──■────■────■──┤ Ry(theta[8]) ├»
     ├──────────────┤├──────────────┤┌─┴─┐  │    │  └──────────────┘»
q_1: ┤ Ry(theta[1]) ├┤ Rz(theta[5]) ├┤ X ├──┼────┼─────────■────────»
     ├──────────────┤├──────────────┤└───┘┌─┴─┐  │       ┌─┴─┐      »
q_2: ┤ Ry(theta[2]) ├┤ Rz(theta[6]) ├─────┤ X ├──┼───────┤ X ├──────»
     ├──────────────┤├──────────────┤     └───┘┌─┴─┐     └───┘      »
q_3: ┤ Ry(theta[3]) ├┤ Rz(theta[7]) ├──────────┤ X ├────────────────»
     └──────────────┘└──────────────┘          └───┘                »
«     ┌───────────────┐                                                       »
«q_0: ┤ Rz(theta[12]) ├─────────────────────────────────────────■──────────■──»
«     └───────────────┘┌──────────────┐┌───────────────┐      ┌─┴─┐        │  »
«q_1: ────────■────────┤ Ry(theta[9]) ├┤ Rz(theta[13]) ├──────┤ X ├────────┼──»
«   

In [29]:
# ============================================================
# Ansatz Parameters
# ============================================================

print("Variational parameters:")
print(ansatz_parameters)

print("\nParameter count:")
print(len(ansatz_parameters))

Variational parameters:
theta, ['theta[0]', 'theta[1]', 'theta[2]', 'theta[3]', 'theta[4]', 'theta[5]', 'theta[6]', 'theta[7]', 'theta[8]', 'theta[9]', 'theta[10]', 'theta[11]', 'theta[12]', 'theta[13]', 'theta[14]', 'theta[15]', 'theta[16]', 'theta[17]', 'theta[18]', 'theta[19]', 'theta[20]', 'theta[21]', 'theta[22]', 'theta[23]', 'theta[24]', 'theta[25]', 'theta[26]', 'theta[27]', 'theta[28]', 'theta[29]', 'theta[30]', 'theta[31]']

Parameter count:
32


In [30]:
# ============================================================
# CELL 8 — VQE Initial Parameters
# ============================================================

def initialize_vqe_parameters(
    num_parameters: int,
    method: str = "random",
    seed: Optional[int] = None,
    initial_parameters: Optional[np.ndarray] = None
) -> np.ndarray:
    """
    Generate the initial variational parameters for VQE.

    Parameters
    ----------
    num_parameters : int
        Number of variational parameters in the ansatz.

    method : str
        Initialization method.

        Supported:
            "random"
            "zeros"

    seed : int, optional
        Random seed used for reproducibility.

    initial_parameters : np.ndarray, optional
        Explicit user-provided initial parameter vector.
        When supplied, this takes precedence over `method`.

    Returns
    -------
    np.ndarray
        Initial parameter vector.
    """

    # --------------------------------------------------------
    # User-provided parameters
    # --------------------------------------------------------

    if initial_parameters is not None:

        parameters = np.asarray(
            initial_parameters,
            dtype=float
        )

        if parameters.ndim != 1:
            raise ValueError(
                "`initial_parameters` must be a "
                "one-dimensional array."
            )

        if len(parameters) != num_parameters:
            raise ValueError(
                f"Expected {num_parameters} initial parameters, "
                f"but received {len(parameters)}."
            )

        return parameters.copy()

    # --------------------------------------------------------
    # Validate initialization method
    # --------------------------------------------------------

    supported_methods = {
        "random",
        "zeros"
    }

    if method not in supported_methods:
        raise ValueError(
            f"Unsupported initialization method: {method}. "
            f"Supported methods: {supported_methods}"
        )

    # --------------------------------------------------------
    # Initialize parameters
    # --------------------------------------------------------

    if method == "zeros":

        parameters = np.zeros(
            num_parameters,
            dtype=float
        )

    elif method == "random":

        rng = np.random.default_rng(seed)

        parameters = rng.uniform(
            low=-np.pi,
            high=np.pi,
            size=num_parameters
        )

    return parameters

In [31]:
# ============================================================
# Initialize Parameters for Current Ansatz
# ============================================================

initial_parameters = initialize_vqe_parameters(
    num_parameters=len(ansatz_parameters),
    method=INITIAL_PARAMETERS,
    seed=RANDOM_SEED
)

print("Initial VQE parameters:")
print(initial_parameters)

print("\nNumber of parameters:")
print(len(initial_parameters))

# ============================================================
# Reproducibility Check
# ============================================================

initial_parameters_check = initialize_vqe_parameters(
    num_parameters=len(ansatz_parameters),
    method=INITIAL_PARAMETERS,
    seed=RANDOM_SEED
)

assert np.allclose(
    initial_parameters,
    initial_parameters_check
)

print("Initial parameter generation is reproducible.")


initial_parameters = initialize_vqe_parameters(
    num_parameters=len(ansatz_parameters),
    method=INITIAL_PARAMETERS,
    seed=RANDOM_SEED
)

Initial VQE parameters:
[ 1.721317 -0.384038  2.253137  1.2401   -2.549859  2.988423  1.640789
  1.797395 -2.336631 -0.311734 -0.8118    2.681444  0.903931  2.027971
 -0.355539 -1.71381   0.342966 -2.740617  2.058567  0.827272  1.621613
 -0.91404   2.957483  2.470053  1.749135 -1.918642 -0.209098 -2.866365
 -2.172163  1.15013   1.537886  2.93745 ]

Number of parameters:
32
Initial parameter generation is reproducible.


In [32]:
# ============================================================
# CELL 9 — VQE Energy Evaluation and Optimization
# ============================================================

def create_vqe_energy_function(
    ansatz: QuantumCircuit,
    parameters: ParameterVector,
    hamiltonian: SparsePauliOp,
    estimator: StatevectorEstimator
) -> Callable[[np.ndarray], float]:
    """
    Create the VQE energy function:

        E(theta) = <psi(theta)|H|psi(theta)>

    The function is compatible with Qiskit's current
    Estimator V2 / StatevectorEstimator interface.
    """

    def energy_function(theta: np.ndarray) -> float:

        theta = np.asarray(theta, dtype=float)

        # Validate parameter count
        if theta.ndim != 1:
            raise ValueError(
                "`theta` must be a one-dimensional array."
            )

        if len(theta) != len(parameters):
            raise ValueError(
                f"Expected {len(parameters)} parameters, "
                f"received {len(theta)}."
            )

        # ----------------------------------------------------
        # Create one PUB:
        #
        # (circuit, observable, parameter values)
        # ----------------------------------------------------

        job = estimator.run(
            [
                (
                    ansatz,
                    hamiltonian,
                    theta
                )
            ]
        )

        pub_result = job.result()[0]

        # ----------------------------------------------------
        # Current Qiskit may return:
        #
        #   scalar / 0-D array
        #
        # for a single expectation value.
        #
        # np.asarray(...).squeeze().item()
        # safely converts that to a Python scalar.
        # ----------------------------------------------------

        energy = np.asarray(
            pub_result.data.evs
        ).squeeze().item()

        return float(
            np.real(energy)
        )

    return energy_function


# ============================================================
# VQE Optimization
# ============================================================

def run_vqe(
    ansatz: QuantumCircuit,
    parameters: ParameterVector,
    hamiltonian: SparsePauliOp,
    initial_parameters: np.ndarray,
    optimizer: str = "COBYLA",
    max_iterations: int = 100,
    tolerance: float = 1e-6,
    seed: Optional[int] = None
) -> Dict[str, Any]:
    """
    Run VQE.

    Important:
        optimizer_iterations = scipy_result.nit
        function_evaluations  = scipy_result.nfev

    These are different quantities.

    VQE minimizes:

        E(theta) = <psi(theta)|H|psi(theta)>
    """

    initial_parameters = np.asarray(
        initial_parameters,
        dtype=float
    )

    if initial_parameters.ndim != 1:
        raise ValueError(
            "`initial_parameters` must be one-dimensional."
        )

    if len(initial_parameters) != len(parameters):
        raise ValueError(
            f"Expected {len(parameters)} initial parameters, "
            f"received {len(initial_parameters)}."
        )

    # --------------------------------------------------------
    # Statevector estimator
    # --------------------------------------------------------

    estimator = StatevectorEstimator(
        seed=seed
    )

    # --------------------------------------------------------
    # Energy function
    # --------------------------------------------------------

    energy_function = create_vqe_energy_function(
        ansatz=ansatz,
        parameters=parameters,
        hamiltonian=hamiltonian,
        estimator=estimator
    )

    # --------------------------------------------------------
    # Optimization history
    # --------------------------------------------------------

    energy_history = []
    parameter_history = []

    # --------------------------------------------------------
    # Objective passed to SciPy
    # --------------------------------------------------------

    def objective(theta):

        energy = energy_function(theta)

        energy_history.append(
            float(energy)
        )

        parameter_history.append(
            np.asarray(
                theta,
                dtype=float
            ).copy()
        )

        return energy

    # --------------------------------------------------------
    # Select optimizer
    # --------------------------------------------------------

    optimizer_upper = optimizer.upper()

    if optimizer_upper == "COBYLA":

        optimizer_method = "COBYLA"

        optimizer_options = {
            "maxiter": max_iterations,
            "tol": tolerance
        }

    elif optimizer_upper == "POWELL":

        optimizer_method = "Powell"

        optimizer_options = {
            "maxiter": max_iterations,
            "xtol": tolerance,
            "ftol": tolerance
        }

    elif optimizer_upper == "NELDER-MEAD":

        optimizer_method = "Nelder-Mead"

        optimizer_options = {
            "maxiter": max_iterations,
            "xatol": tolerance,
            "fatol": tolerance
        }

    elif optimizer_upper == "L-BFGS-B":

        optimizer_method = "L-BFGS-B"

        optimizer_options = {
            "maxiter": max_iterations,
            "ftol": tolerance,
            "gtol": tolerance,
            "maxls": 50
        }

    else:

        raise ValueError(
            f"Unsupported optimizer: {optimizer}. "
            "Supported optimizers are: "
            "COBYLA, POWELL, NELDER-MEAD, L-BFGS-B."
        )

    # --------------------------------------------------------
    # Run optimization
    # --------------------------------------------------------

    optimization_result = minimize(
        fun=objective,
        x0=initial_parameters,
        method=optimizer_method,
        options=optimizer_options
    )

    # --------------------------------------------------------
    # Final parameters and energy
    # --------------------------------------------------------

    optimal_parameters = np.asarray(
        optimization_result.x,
        dtype=float
    )

    minimum_energy = energy_function(
        optimal_parameters
    )

    # --------------------------------------------------------
    # Correct optimizer statistics
    # --------------------------------------------------------

    optimizer_iterations = getattr(
        optimization_result,
        "nit",
        None
    )

    function_evaluations = getattr(
        optimization_result,
        "nfev",
        None
    )

    # --------------------------------------------------------
    # Return structured results
    # --------------------------------------------------------

    return {
        "optimal_parameters":
            optimal_parameters,

        "minimum_energy":
            float(minimum_energy),

        "energy_history":
            energy_history,

        "parameter_history":
            parameter_history,

        # TRUE optimizer iterations
        "iterations":
            int(optimizer_iterations)
            if optimizer_iterations is not None
            else None,

        # Number of times objective was evaluated
        "function_evaluations":
            int(function_evaluations)
            if function_evaluations is not None
            else len(energy_history),

        "success":
            bool(
                optimization_result.success
            ),

        "message":
            str(
                optimization_result.message
            ),

        "optimizer_result":
            optimization_result,

        "estimator":
            estimator
    }

In [33]:
# ============================================================
# Run VQE
# ============================================================

vqe_results = run_vqe(
    ansatz=ansatz,
    parameters=ansatz_parameters,
    hamiltonian=cost_hamiltonian,
    initial_parameters=initial_parameters,
    optimizer=OPTIMIZER,
    max_iterations=MAX_ITERATIONS,
    tolerance=VQE_TOLERANCE,
    seed=RANDOM_SEED
)

print("=" * 60)
print("VQE OPTIMIZATION")
print("=" * 60)

print(f"Optimizer           : {OPTIMIZER}")
print(f"Iterations           : {vqe_results['iterations']}")
print(
    f"Minimum energy       : "
    f"{vqe_results['minimum_energy']:.10f}"
)
print(
    f"Optimization success : "
    f"{vqe_results['success']}"
)
print(
    f"Message              : "
    f"{vqe_results['message']}"
)

VQE OPTIMIZATION
Optimizer           : COBYLA
Iterations           : None
Minimum energy       : -0.0127575934
Optimization success : False
Message              : Return from COBYLA because the objective function has been evaluated MAXFUN times.


In [34]:
# ============================================================
# VQE Result Analysis
# ============================================================

def analyze_vqe_results(
    ansatz: QuantumCircuit,
    optimal_parameters: np.ndarray,
    problem: PortfolioProblem,
    hamiltonian: SparsePauliOp
) -> pd.DataFrame:
    """
    Analyze every computational-basis state of the
    optimized VQE state.

    The quantum probability comes from:

        P(x) = |<x|psi(theta*)>|^2

    The Hamiltonian energy is calculated independently
    for each computational-basis state.

    No portfolio combination is hard-coded.
    """

    n = problem.num_assets

    # --------------------------------------------------------
    # Optimized circuit
    # --------------------------------------------------------

    optimized_circuit = ansatz.assign_parameters(
        optimal_parameters,
        inplace=False
    )

    # --------------------------------------------------------
    # Optimized statevector
    # --------------------------------------------------------

    statevector = Statevector.from_instruction(
        optimized_circuit
    )

    probabilities = (
        np.abs(statevector.data) ** 2
    )

    # --------------------------------------------------------
    # Hamiltonian matrix
    # --------------------------------------------------------

    H_matrix = hamiltonian.to_matrix()

    results = []

    # --------------------------------------------------------
    # Generate all 2^N states
    # --------------------------------------------------------

    for bits in product(
        [0, 1],
        repeat=n
    ):

        x = np.array(
            bits,
            dtype=int
        )

        bitstring = "".join(
            map(str, x)
        )

        # ----------------------------------------------------
        # Qiskit basis ordering
        # ----------------------------------------------------

        qiskit_bitstring = bitstring[::-1]

        basis_index = int(
            qiskit_bitstring,
            2
        )

        # ----------------------------------------------------
        # Quantum probability
        # ----------------------------------------------------

        probability = float(
            probabilities[basis_index]
        )

        # ----------------------------------------------------
        # Computational basis vector
        # ----------------------------------------------------

        basis_state = np.zeros(
            2 ** n,
            dtype=complex
        )

        basis_state[basis_index] = 1.0

        # ----------------------------------------------------
        # Hamiltonian energy
        # ----------------------------------------------------

        energy = float(
            np.vdot(
                basis_state,
                H_matrix @ basis_state
            ).real
        )

        # ----------------------------------------------------
        # Classical portfolio quantities
        # ----------------------------------------------------

        portfolio_data = calculate_portfolio_cost(
            x=x,
            problem=problem
        )

        # ----------------------------------------------------
        # Asset names
        # ----------------------------------------------------

        selected_assets = [
            problem.asset_names[i]
            for i, bit in enumerate(x)
            if bit == 1
        ]

        selected_assets_text = (
            ", ".join(selected_assets)
            if selected_assets
            else "None"
        )

        # ----------------------------------------------------
        # Feasibility
        # ----------------------------------------------------

        feasible = bool(
            np.isclose(
                portfolio_data["investment"],
                problem.budget
            )
        )

        results.append({
            "bitstring":
                bitstring,

            "selected_assets":
                selected_assets_text,

            "probability":
                probability,

            "energy":
                energy,

            "expected_return":
                portfolio_data[
                    "expected_return"
                ],

            "risk":
                portfolio_data[
                    "risk"
                ],

            "investment":
                portfolio_data[
                    "investment"
                ],

            "budget_difference":
                portfolio_data[
                    "budget_difference"
                ],

            "objective":
                portfolio_data[
                    "total_cost"
                ],

            "selected_count":
                portfolio_data[
                    "selected_count"
                ],

            "feasible":
                feasible
        })

    return pd.DataFrame(
        results
    )


# ============================================================
# Run analysis
# ============================================================

vqe_state_results = analyze_vqe_results(
    ansatz=ansatz,
    optimal_parameters=
        vqe_results["optimal_parameters"],
    problem=portfolio,
    hamiltonian=cost_hamiltonian
)


# ============================================================
# Display all combinations
# ============================================================

print("=" * 90)
print("VQE PROBABILITY DISTRIBUTION")
print("=" * 90)

display(
    vqe_state_results.sort_values(
        by="probability",
        ascending=False
    ).reset_index(drop=True)
)


# ============================================================
# VQE PREDICTION
# ============================================================
#
# IMPORTANT:
#
# The prediction comes from the optimized quantum state,
# NOT from the classical exhaustive-search energy ranking.
#
# We first restrict to feasible states, then select the
# feasible state with the highest VQE probability.
# ============================================================

feasible_vqe_results = (
    vqe_state_results[
        vqe_state_results["feasible"]
    ]
    .sort_values(
        by="probability",
        ascending=False
    )
    .reset_index(drop=True)
)

if len(feasible_vqe_results) == 0:

    raise ValueError(
        "VQE produced no feasible portfolio states."
    )


best_vqe_prediction = (
    feasible_vqe_results.iloc[0]
)


# ============================================================
# Print VQE prediction
# ============================================================

print("=" * 90)
print("VQE PREDICTION")
print("=" * 90)

print(
    f"Bitstring       : "
    f"{best_vqe_prediction['bitstring']}"
)

print(
    f"Selected assets : "
    f"{best_vqe_prediction['selected_assets']}"
)

print(
    f"Probability     : "
    f"{best_vqe_prediction['probability']:.10f}"
)

print(
    f"Hamiltonian energy : "
    f"{best_vqe_prediction['energy']:.10f}"
)

print(
    f"Investment      : "
    f"{best_vqe_prediction['investment']:.6f}"
)

print(
    f"Expected return : "
    f"{best_vqe_prediction['expected_return']:.6f}"
)

print(
    f"Risk            : "
    f"{best_vqe_prediction['risk']:.6f}"
)

print(
    f"Feasible        : "
    f"{best_vqe_prediction['feasible']}"
)


# ============================================================
# Classical vs VQE
# ============================================================

print("=" * 90)
print("CLASSICAL vs VQE")
print("=" * 90)

print(
    f"Classical optimum : "
    f"{classical_optimum['bitstring']}"
)

print(
    f"Classical assets  : "
    f"{classical_optimum['selected_assets']}"
)

print(
    f"Classical energy/objective : "
    f"{classical_optimum['objective']:.10f}"
)

print()

print(
    f"VQE prediction    : "
    f"{best_vqe_prediction['bitstring']}"
)

print(
    f"VQE assets        : "
    f"{best_vqe_prediction['selected_assets']}"
)

print(
    f"VQE state energy  : "
    f"{best_vqe_prediction['energy']:.10f}"
)

print(
    f"VQE probability   : "
    f"{best_vqe_prediction['probability']:.10f}"
)

print()

if (
    best_vqe_prediction["bitstring"]
    ==
    classical_optimum["bitstring"]
):

    print(
        "RESULT: VQE FOUND THE CLASSICAL OPTIMAL COMBINATION."
    )

else:

    print(
        "RESULT: VQE DID NOT FIND THE CLASSICAL "
        "OPTIMAL COMBINATION IN THIS RUN."
    )

VQE PROBABILITY DISTRIBUTION


,bitstring,selected_assets,probability,energy,expected_return,risk,investment,budget_difference,objective,selected_count,feasible
0,1011,"A, C, D",0.99890229,-0.18900000,0.30000000,0.11100000,5.00000000,0.00000000,-0.18900000,3,True
1,1110,"A, B, C",0.00055864,99.89500000,0.35000000,0.24500000,6.00000000,1.00000000,99.89500000,3,False
2,0111,"B, C, D",0.00024013,99.87300000,0.34000000,0.21300000,6.00000000,1.00000000,99.87300000,3,False
3,0010,C,0.00015816,399.86500000,0.15000000,0.01500000,3.00000000,-2.00000000,399.86500000,1,False
4,0011,"C, D",0.00007693,99.83200000,0.22000000,0.05200000,4.00000000,-1.00000000,99.83200000,2,False
5,1010,"A, C",0.00004286,99.84000000,0.23000000,0.07000000,4.00000000,-1.00000000,99.84000000,2,False
6,1000,A,0.00000842,1599.96000000,0.08000000,0.04000000,1.00000000,-4.00000000,1599.96000000,1,False
7,1101,"A, B, D",0.00000571,99.90000000,0.27000000,0.17000000,4.00000000,-1.00000000,99.90000000,3,False
8,1001,"A, D",0.00000188,899.92400000,0.15000000,0.07400000,2.00000000,-3.00000000,899.92400000,2,False
9,0100,B,0.00000167,899.95000000,0.12000000,0.07000000,2.00000000,-3.00000000,899.95000000,1,False


VQE PREDICTION
Bitstring       : 1011
Selected assets : A, C, D
Probability     : 0.9989022905
Hamiltonian energy : -0.1890000000
Investment      : 5.000000
Expected return : 0.300000
Risk            : 0.111000
Feasible        : True
CLASSICAL vs VQE
Classical optimum : 1011
Classical assets  : A, C, D
Classical energy/objective : -0.1890000000

VQE prediction    : 1011
VQE assets        : A, C, D
VQE state energy  : -0.1890000000
VQE probability   : 0.9989022905

RESULT: VQE FOUND THE CLASSICAL OPTIMAL COMBINATION.
